# AdvisorAudit sentence-localization visualizer

Load localization outputs, inspect deception-rate trajectories over sentence prefixes, and manually verify sampled generations for advisor recommendations.

In [ ]:
from pathlib import Path
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
MODEL_TAG = "DeepSeek-R1-Distill-Qwen-7B"  # change as needed
data_dir = Path(f"/playpen-ssd/smerrill/deception2/AdvisorAudit/Results/SentencePipeline/v1/{MODEL_TAG}")
json_dir = data_dir / "localization"
examples_path = data_dir / "examples.jsonl"

json_files = sorted(json_dir.glob("sentence_localization_*.json"))
jsonl_files = sorted(data_dir.glob("localization*.jsonl"))

print(f"data_dir: {data_dir}")
print(f"Found {len(json_files)} per-example JSON files")
print(f"Found {len(jsonl_files)} localization JSONL files")
print(f"examples_path exists: {examples_path.exists()}")

In [ ]:
def load_json(path: Path):
    return json.loads(path.read_text(encoding="utf-8"))

def load_jsonl(path: Path):
    with path.open("r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            yield json.loads(line)

def load_result_from_jsonl(path: Path, example_id=None, index=0):
    for i, rec in enumerate(load_jsonl(path)):
        if example_id is not None:
            if rec.get("example_id") == example_id:
                return rec
        elif i == index:
            return rec
    raise ValueError("No matching record found")

def build_examples_lookup(path: Path):
    if not path.exists():
        return {}
    out = {}
    for rec in load_jsonl(path):
        example_id = rec.get("example_id")
        if example_id:
            out[example_id] = rec
    return out

examples_lookup = build_examples_lookup(examples_path)
print(f"Loaded {len(examples_lookup)} examples")

## Load one localization record

Use either per-example JSON (`json_files`) or a JSONL record.

In [ ]:
result = None

# Option A: pick by file index from per-example JSON outputs
json_file_idx = 0  # change this
if json_files:
    idx = min(max(json_file_idx, 0), len(json_files) - 1)
    result = load_json(json_files[idx])
    print(f"Loaded JSON: {json_files[idx]}")

# Option B: uncomment to load from JSONL
# if jsonl_files:
#     jsonl_path = jsonl_files[0]
#     result = load_result_from_jsonl(jsonl_path, index=0)
#     print(f"Loaded JSONL: {jsonl_path}")

In [ ]:
if result is None:
    raise RuntimeError("No result loaded")

example_id = result.get("example_id")
history = result.get("history", [])
eval_context = result.get("eval_context", {})
full_score = result.get("full_score", {}) or {}

print(f"example_id: {example_id}")
print(f"game: {result.get('game')}")
print(f"history probes: {len(history)}")
print("eval_context:", eval_context)

ex = examples_lookup.get(example_id, {})
if ex:
    print(f"example deceptive label: {ex.get('deceptive')}")
    print("truth_context:", ex.get("truth_context"))

if full_score:
    print(f"full_score deception_rate: {full_score.get('deception_rate')}")

In [ ]:
rows = []
for i, h in enumerate(history):
    sent_end = h.get("sentence_end_idx")
    if sent_end is None:
        sent_end = h.get("sentence_idx")
    rows.append(
        {
            "step": i,
            "sentence_end_idx": sent_end,
            "deception_rate": h.get("deception_rate"),
            "num_truthful": h.get("num_truthful"),
            "num_valid": h.get("num_valid"),
            "ci_low": h.get("ci_low"),
            "ci_high": h.get("ci_high"),
        }
    )

df_hist = pd.DataFrame(rows).sort_values("sentence_end_idx")
df_hist

In [ ]:
plt.figure(figsize=(10, 4))
plt.plot(df_hist["sentence_end_idx"], df_hist["deception_rate"], marker="o")

if df_hist["ci_low"].notna().any():
    plt.fill_between(
        df_hist["sentence_end_idx"],
        df_hist["ci_low"],
        df_hist["ci_high"],
        alpha=0.2,
        label="95% CI",
    )

plt.axhline(0.5, linestyle="--", linewidth=1)
plt.xlabel("Sentence index (prefix end)")
plt.ylabel("Deception rate")
plt.title("AdvisorAudit sentence-level deception localization")
plt.grid(alpha=0.3)
plt.show()

## Probe inspection

Inspect sampled generations and recovered recommendation IDs.

In [ ]:
probe_idx = 0  # change this
probe = history[probe_idx]

print(f"probe_idx: {probe_idx}")
print(f"sentence_end_idx: {probe.get('sentence_end_idx')} | deception_rate: {probe.get('deception_rate')}")
print(f"char_span: {probe.get('char_span')}")
print(f"sentence_text: {probe.get('sentence_text')}")

gens = probe.get("generations", [])
print(f"Generations: {len(gens)}")

gen_rows = []
for i, g in enumerate(gens):
    ev = g.get("evaluation") or {}
    gen_rows.append(
        {
            "gen_idx": i,
            "is_truthful": g.get("is_truthful"),
            "deceptive": g.get("deceptive"),
            "recommended_option_id": ev.get("recommended_option_id"),
            "best_option_id": ev.get("best_option_id"),
            "recommendation_source": ev.get("recommendation_source"),
            "parse_error": g.get("parse_error"),
        }
    )

df_gen = pd.DataFrame(gen_rows)
display(df_gen.head(20))
if not df_gen.empty:
    display(df_gen["recommended_option_id"].value_counts(dropna=False).rename("count"))

In [ ]:
max_print = 3
for i, g in enumerate(gens[:max_print]):
    print(f"\n=== Generation {i} ===\n")
    print(g.get("gen_text", ""))

## Aggregate summary (optional)

Loads records from localization JSONL (preferred) or per-example JSONs.

In [ ]:
records = []
if jsonl_files:
    records = list(load_jsonl(jsonl_files[0]))
elif json_files:
    records = [load_json(p) for p in json_files]

rows = []
for rec in records:
    hist = rec.get("history", [])
    final_rate = None
    if rec.get("full_score") and isinstance(rec.get("full_score"), dict):
        final_rate = rec["full_score"].get("deception_rate")
    if final_rate is None and hist:
        final_rate = hist[-1].get("deception_rate")
    ctx = rec.get("eval_context", {}) or {}
    rows.append(
        {
            "example_id": rec.get("example_id"),
            "game": rec.get("game"),
            "num_probes": len(hist),
            "final_deception_rate": final_rate,
            "best_option_id": ctx.get("best_option_id"),
            "high_commission_option_id": ctx.get("high_commission_option_id"),
        }
    )

df_summary = pd.DataFrame(rows)
print(f"Loaded {len(df_summary)} localization records")
display(df_summary.head())

if not df_summary.empty:
    print(df_summary["final_deception_rate"].describe())
    plt.figure(figsize=(7, 4))
    plt.hist(df_summary["final_deception_rate"].dropna(), bins=20)
    plt.xlabel("Final deception rate")
    plt.ylabel("Count")
    plt.title("AdvisorAudit localization final deception-rate distribution")
    plt.grid(alpha=0.3)
    plt.show()